# ANALYSIS
---
## DOCUMENTATION
- Consideration of Inbound vs Outbound assumes independence of the data sets and completely neglects the context of the datasets so that approach is to be sidelined and perhaps studied in the future. Instead, consideration of the direction categorical variable will only be used to measure flow for each company.
- Some Assumptions are made in regards to the data
    1. Duplicate Values are rejected and considered system errors. This assumption is justified by the very small percentage of duplicates relative to the data set size.
    2. The three data sets hold equal weighting in the calculation of Share of Wallet and therefore are aggregated and considered in their entirety. The justification may simply be to model the entirety of our data due to the sheer scale of our capture. Doing this allows for a more precise measurement but sacrifices fitting well for predictive values, as such, this model is only used to understand the current data while we rely heavily on inference and business insight for our ranking system

## Phase Splitting for Analysis Section
- The Analysis is now split into phases that shall be detailed below
  ### Phase 1: Data Verification and Preliminary Data Exploration
  - **Completed**
  - Data importing
  - Analysis of Observations present
  - Ensuring certain criteria are met to ensure data validity
  - Manipulating the data into a usable format
  ### Phase 2: Data Aggregation and Internal Capture
  - **Completed**
  - Data aggregation via certain key variables
  - Assumptions stated with motivation regarding the variable selection
  - Final calculation of Total Internal Capture per company
  ### Phase 3: External Capture and Text Mining via Generative AI
  - **In Progress**
  - Retrieval of publicly accessible records via yfinance and parsing of financial records via GenAI
  - Predictive Model Training
  - Predictive Model Testing and Tweaks
  - Final Calculation of Estimated Customer Wallet per company
  ### Conclusion and Results
  - **Incomplete**
  - Estimated Total Wallet Share per Company provided
  - Implementation of Ranking Algorithm pushed to GenAI for interpretation
  - Interactive plots featuring some useful information to be sent to dashboard
  - Statement of all assumptions to be sent to dashboard
  - Technical rundown of model choice and key bottlenecks


## PHASE 1: Data Verification and Preliminary Data Exploration

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib as mp
import matplotlib.pyplot as plt
import plotly.express as px
%matplotlib inline

In [2]:
# Loading the given Data Sets
transactional_data = pd.read_csv("../Data/transactional_banking.csv")
cross_border_payments = pd.read_csv("../Data/cross_border_payments.csv")
trade_finance = pd.read_csv("../Data/trade_finance.csv")

# Grouping the datasets together
ds = {
    "Transactional" : transactional_data,
    "SWIFT" : cross_border_payments,
    "Trade" : trade_finance,
}

In [3]:
# Understanding observations present in the dataset
print(transactional_data.columns)
print("#" + "="*100 + "#")
transactional_data['amount_zar'] = pd.to_numeric(transactional_data['amount_zar'])
print(cross_border_payments.columns)
print("#" + "="*100 + "#")
print(trade_finance.columns)

Index(['transaction_id', 'entity_id', 'entity_name', 'sector', 'date',
       'leg_type', 'direction', 'amount_zar', 'currency', 'channel',
       'beneficiary_name', 'reference', 'memo'],
      dtype='object')
#====================================================================================================#
Index(['transaction_id', 'entity_id', 'entity_name', 'sector', 'date',
       'direction', 'currency_pair', 'value_zar', 'counterparty_country',
       'corridor_type', 'beneficiary_name', 'reference', 'memo'],
      dtype='object')
#====================================================================================================#
Index(['instrument_id', 'entity_id', 'entity_name', 'sector', 'date',
       'instrument_type', 'direction', 'tenor_days', 'value_zar',
       'counterparty_country', 'commodity_or_contract_type', 'status',
       'beneficiary_name', 'reference', 'memo'],
      dtype='object')


In [4]:
def check_entity_alignment(ds_dict):
    """
    A function that ensures all datasets capture the same 
    companies/entities
    """
    all_entities = pd.DataFrame()
    
    for name, df in ds_dict.items():
        unique_mappings = df[['entity_id', 'entity_name', 'sector']].drop_duplicates()
        unique_mappings['source'] = name
        all_entities = pd.concat([all_entities, unique_mappings])

    mismatches = all_entities.groupby('entity_id').nunique()
    inconsistent_ids = mismatches[(mismatches['entity_name'] > 1) | (mismatches['sector'] > 1)].index
    
    if len(inconsistent_ids) > 0:
        print(f"Found inconsistencies for entity IDs: {list(inconsistent_ids)}")
        print(all_entities[all_entities['entity_id'].isin(inconsistent_ids)].sort_values('entity_id'))
    else:
        print("All entity IDs exist and are captured across all datasets.")
        print(f"Total Unique Entities: {all_entities['entity_id'].nunique()}\n")

check_entity_alignment(ds)

All entity IDs exist and are captured across all datasets.
Total Unique Entities: 20



In [5]:
def check_date_alignment(ds_dict):
    """
    Ensures all bank activity is captured without missing data
    """
    for name, df in ds_dict.items():
        # Conversion from regular dtype to datetime format
        df['date'] = pd.to_datetime(df['date'])
        
        min_date = df['date'].min()
        max_date = df['date'].max()
        total_months = df['date'].dt.to_period('M').nunique()
        
        print(f"{name} Data:")
        print(f"  Range: {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")
        print(f"  Total Active Months: {total_months}")
    print("\n")

check_date_alignment(ds)

Transactional Data:
  Range: 2023-07-01 to 2026-06-30
  Total Active Months: 36
SWIFT Data:
  Range: 2023-07-01 to 2026-06-30
  Total Active Months: 36
Trade Data:
  Range: 2023-07-01 to 2026-06-30
  Total Active Months: 36




In [6]:
# Caution, running this cell block takes a few seconds as it is unoptimised, please allow it to run even if no output is produced

def verify_critical_data_integrity(ds_dict):
    """
    A function that finds Duplicate and Missing Values
    """
    text_columns = ['beneficiary_name', 'reference', 'memo']
    
    for name, df in ds_dict.items():
        print(f"[{name} Data]")
        
        # Check for duplicates
        duplicate_count = df.duplicated().sum()
        print(f"  Duplicate Rows: {duplicate_count} ({(duplicate_count/len(df))*100:.2f}%)")
        
        # Check missing values for critical fields
        for col in text_columns:
            if col in df.columns:
                missing_count = df[col].isnull().sum()
                missing_pct = (missing_count / len(df)) * 100
                print(f"  Missing '{col}': {missing_count} rows ({missing_pct:.2f}%)")
        print("-" * 30)

verify_critical_data_integrity(ds)
# if there are duplicates, we can just drop them by the assumption stated in the DOCUMENTATION section above:
for name, df in ds.items():
    ds[name] = df.drop_duplicates()

[Transactional Data]
  Duplicate Rows: 10812 (0.39%)
  Missing 'beneficiary_name': 0 rows (0.00%)
  Missing 'reference': 0 rows (0.00%)
  Missing 'memo': 2799218 rows (99.87%)
------------------------------
[SWIFT Data]
  Duplicate Rows: 926 (0.38%)
  Missing 'beneficiary_name': 0 rows (0.00%)
  Missing 'reference': 0 rows (0.00%)
  Missing 'memo': 240669 rows (99.81%)
------------------------------
[Trade Data]
  Duplicate Rows: 88 (0.43%)
  Missing 'beneficiary_name': 0 rows (0.00%)
  Missing 'reference': 0 rows (0.00%)
  Missing 'memo': 20209 rows (99.54%)
------------------------------


## PHASE 2: Data Aggregation and Internal Capture

In [7]:
def aggregate_transactional(df):
    # Total flow by direction (credit vs debit)
    trans_flows = df.groupby(['entity_id', 'direction'])['amount_zar'].sum().unstack(fill_value=0)
    trans_flows['Total_Transactional_Volume'] = trans_flows.sum(axis=1)
    
    # Breakdown by channel and leg_type
    trans_channels = df.groupby(['entity_id', 'channel', 'leg_type'])['amount_zar'].sum().unstack(fill_value=0)
    
    return trans_flows, trans_channels


def aggregate_swift(df):
    # Total volume by direction
    swift_flows = df.groupby(['entity_id', 'direction'])['value_zar'].sum().unstack(fill_value=0)
    swift_flows['Total_SWIFT_Volume'] = swift_flows.sum(axis=1)
    
    # Breakdown by currency pair and country corridor
    swift_corridors = df.groupby(['entity_id', 'currency_pair', 'counterparty_country'])['value_zar'].sum().unstack(fill_value=0)
    
    return swift_flows, swift_corridors


def aggregate_trade(df):
    # Active vs Expired Exposure by Instrument Type
    trade_exposure = df.groupby(['entity_id', 'instrument_type', 'status'])['value_zar'].sum().unstack(fill_value=0)
    
    # Total value and average tenor by directional mix (Import/Export)
    trade_mix = df.groupby(['entity_id', 'direction']).agg(
        Total_Value=('value_zar', 'sum'),
        Avg_Tenor_Days=('tenor_days', 'mean')
    ).unstack(fill_value=0)
    
    return trade_exposure, trade_mix


trans_flows, trans_channels = aggregate_transactional(ds['Transactional'])
swift_flows, swift_corridors = aggregate_swift(ds['SWIFT'])
trade_exposure, trade_mix = aggregate_trade(ds['Trade'])


client_capture_summary = pd.concat([
    trans_flows['Total_Transactional_Volume'],
    swift_flows['Total_SWIFT_Volume'],
    trade_exposure.sum(axis=1).rename('Total_Trade_Exposure')
], axis=1).fillna(0)

# Calculation of the total Syn Bank wallet currently captured per company
client_capture_summary['Total_SynBank_Capture_ZAR'] = client_capture_summary.sum(axis=1)

### THIS SECTION IS MERELY FOR STYLE ###

# Scaling the entire dataframe by 1 Billion
summary_in_billions = client_capture_summary / 1e9

# Renaming columns
summary_in_billions = summary_in_billions.rename(columns={
    'Total_Transactional_Volume': 'Transactional (ZARbn)',
    'Total_SWIFT_Volume': 'SWIFT (ZARbn)',
    'Total_Trade_Exposure': 'Trade (ZARbn)',
    'Total_SynBank_Capture_ZAR': 'Total Capture (ZARbn)'
})

# Extract the unique mapping of entity_id to entity_name from the transactional data
entity_mapping = ds['Transactional'][['entity_id', 'entity_name']].drop_duplicates().set_index('entity_id')

summary_with_names = summary_in_billions.join(entity_mapping)
summary_with_names = summary_with_names.set_index('entity_name')

# Display the top 5
print("# ====== Top 5 Clients by Total Captured Volume (ZARbn) ====== #")
print(summary_with_names.sort_values('Total Capture (ZARbn)', ascending=False).head())

# ====== Top 5 Clients by Total Captured Volume (ZARbn) ====== #
                 Transactional (ZARbn)  SWIFT (ZARbn)  Trade (ZARbn)  \
entity_name                                                            
Pepkor Holdings             103.040428      19.993871            0.0   
Sanlam                       73.844810       8.341689            0.0   
BHP Group                    46.526131       6.340580            0.0   
MTN Group                    31.404511      19.316555            0.0   
Bid Corporation              33.248738      15.311634            0.0   

                 Total Capture (ZARbn)  
entity_name                             
Pepkor Holdings             123.034299  
Sanlam                       82.186499  
BHP Group                    52.866711  
MTN Group                    50.721066  
Bid Corporation              48.560372  


## PHASE 3: External Capture and Text Mining via Generative AI

In [13]:
# Extracting Mined Data from json provided by the GenAI segment

# json format and information needed

# '''[
#   {
#     "entity_name": ,
#     "revenue": ,
#     "cost_of_sales": ,
#     "foreign_costs_imports": ,
#     "net_worth": ,
#     "total_debt": ,
#     "total_liquidity": 
#   }
# ]'''

external_df = pd.read_json('external_financials.json')

# Assumed weighting values to be discussed below
ALPHA =   # Transactional (Revenue + Cost of Sales)
BETA  =   # SWIFT/FX processing (e.g 100% of Foreign Imports)
GAMMA =   # Trade Finance proxy (e.g 25% of foreign trade backed by LC/Guarantees)
DELTA =   # Trade Finance proxy (e.g 5% of total cos backed by supply chain finance)
EPSILON =  # Lending proxy (e.g 100% of debt servicing capacity)

external_df['Est_Trans_Wallet_ZARbn'] = (ALPHA * (external_df['revenue'] + external_df['cost_of_sales'])) / 1e9
external_df['Est_SWIFT_Wallet_ZARbn'] = (BETA * external_df['foreign_costs_imports']) / 1e9
external_df['Est_Trade_Wallet_ZARbn'] = (GAMMA * external_df['foreign_costs_imports'] + DELTA * external_df['cost_of_sales']) / 1e9
external_df['Est_Lending_Wallet_ZARbn'] = (EPSILON * external_df['total_debt']) / 1e9

external_df['Total_Estimated_Wallet_ZARbn'] = (
    external_df['Est_Trans_Wallet_ZARbn'] +
    external_df['Est_SWIFT_Wallet_ZARbn'] +
    external_df['Est_Trade_Wallet_ZARbn'] +
    external_df['Est_Lending_Wallet_ZARbn']
)

wallet_estimates = external_df[[
    'entity_name', 'Est_Trans_Wallet_ZARbn', 'Est_SWIFT_Wallet_ZARbn', 
    'Est_Trade_Wallet_ZARbn', 'Est_Lending_Wallet_ZARbn', 'Total_Estimated_Wallet_ZARbn'
]]

FileNotFoundError: File external_financials.json does not exist

In [ ]:
# Aggregation of Internal Capture and External
sow_df = summary_with_names.reset_index().merge(
    wallet_estimates, 
    on='entity_name', 
    how='left'
)

# Calculation of Share of Wallet
sow_df['Share_of_Wallet_%'] = (
    sow_df['Total Capture (ZARbn)'] / sow_df['Total_Estimated_Wallet_ZARbn']
) * 100

# Calculation of Revenue Gap / Competitor Leakage
sow_dataframe['Competitor_Leakage_ZARbn'] = (
    sow_dataframe['Total_Estimated_Wallet_ZARbn'] - sow_dataframe['Total Capture (ZARbn)']
)

# Sort by biggest commercial opportunity
sow_dataframe = sow_dataframe.sort_values('Competitor_Leakage_ZARbn', ascending=False)

pd.options.display.float_format = '{:,.2f}'.format
print("# ====== Top Clients Ranked by Growth Opportunity (Competitor Leakage) ====== #")
print(sow_dataframe[[
    'entity_name', 'Total Capture (ZARbn)', 
    'Total_Estimated_Wallet_ZARbn', 'Share_of_Wallet_%', 'Competitor_Leakage_ZARbn'
]].head())

## Assumptions, Bottlenecks and Improvements
- Multipiers (alpha, beta, gamma, delta and epsilon)
    - we think of these as the assumptions we make of a companies total business in a particular sector
    - these multipliers can be changed by a GenAI model to better reflect future trends, this allows the model to fit well for future rankings as more data is collected
    - A good improvement to be made especially for longer term usage would be to implement a database of sorts to store historical data rather than continuosly fetching/scraping data from the internet.

## Plots

## Conclusion